## **Eval of Finetuned Qwen3-8B with a Cross-Encoder**


In [ ]:
%pip install transformers accelerate bitsandbytes
%pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu130

Miscellaneous

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

Define a function to load the two models (base and finetuned). Run one after the other.

In [3]:
def load_model(model_name):
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        quantization_config=quant_config,
    )
    return tokenizer, model

Define function to prompt

In [4]:
def prompt_model(tokenizer, model, prompt, max_new_tokens=500):
    messages = [
        {"role": "system", "content": "Rewrite the user's text in Gen-Z brainrot slang. Keep the meaning."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )

    generated = outputs[0][inputs.input_ids.shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

Setting up evaluator (cross encoder)

In [5]:
from sentence_transformers import util, CrossEncoder # SentenceTransformer

# evaluator = SentenceTransformer("all-MiniLM-L6-v2")
evaluator = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

def semantic_score(reference, prediction):
    score = evaluator.predict([reference, prediction])
    return score.item()

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Read test lines

In [8]:
import json
data = []
with open("test.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

Run base and finetuned model on test prompts and compare to test answers

In [9]:
tokenizer, base_model = load_model("Qwen/Qwen3-4B")
base_answers = [prompt_model(
    tokenizer,
    base_model,
    item["source"]
) for item in data]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [17]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
	repo_id="PrimitivePenguin/brainrot-qwen3-4b-GGUF",
	filename="brainrot.gguf",
)

finetuned_answers = [llm.create_chat_completion(
	messages = [
			 {
						"role": "system",
			  		"content": "Rewrite the user's text in Gen-Z brainrot slang. Keep the meaning."
			 },
			 {
						"role": "user",
			  		"content": item["source"]
			 }]
)['choices'][0]['message']['content'][19:] for item in data]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
llama_model_loader: loaded meta data with 30 key-value pairs and 398 tensors from /root/.cache/huggingface/hub/models--PrimitivePenguin--brainrot-qwen3-4b-GGUF/snapshots/1f35f34766a4eb5942bde53a9b101686e83aea04/./brainrot.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.9

In [40]:
base_scores = [semantic_score(
    base_answers[i], data[i]["target"]
) for i in range(len(data))]
finetuned_scores = [semantic_score(
    finetuned_answers[i], data[i]["target"]
) for i in range(len(data))]


In [43]:
print(base_scores)
print(finetuned_scores)
print(sum(base_scores[i] < finetuned_scores[i] for i in range(len(data)))/len(data))

[-4.280340194702148, -3.1841835975646973, -6.858484268188477, -4.195919036865234, -6.368988990783691, -6.1164469718933105, -7.1487321853637695, -5.412529468536377, -3.588351249694824, -11.059549331665039, -5.656313896179199, -9.042625427246094, 6.382845878601074, -8.79100513458252, -5.0293378829956055, -9.248188018798828, -11.370377540588379, -6.581939697265625, -5.192442893981934, -6.226431369781494, -6.675206661224365, -2.305246353149414, -5.20357608795166, -6.616949558258057, -11.392807960510254, -10.283883094787598, -11.038416862487793, 5.809130668640137, -10.45991039276123, 1.0328855514526367, -11.252204895019531, -10.058863639831543, -2.023801326751709, -10.308906555175781, 1.3280680179595947]
[2.1084980964660645, -1.8540440797805786, -0.8264787197113037, -2.865804672241211, -9.870655059814453, -9.888840675354004, -10.822351455688477, -11.35106086730957, -2.189422607421875, -11.233394622802734, -7.100079536437988, -11.190231323242188, 2.8686928749084473, -0.864396333694458, -1.45